[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module5/09-ml-production.ipynb)

# Module 5 — Lesson 9: ML in Production

**Module:** 5 — Machine Learning Foundations | **Time:** 30 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Save and load scikit-learn models with `joblib` and `pickle`
- Serialize entire `Pipeline` objects for safe deployment
- Track experiments with MLflow (parameters, metrics, and model artifacts)
- Explain the concept of data drift and detect it using the KS test and PSI
- Write a minimal FastAPI serving endpoint for a trained model
- Understand the structure of a production Dockerfile

In [ ]:
!pip install -q mlflow evidently

import os
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy import stats

import mlflow
import mlflow.sklearn

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# Working directory for saved artefacts
ARTIFACT_DIR = Path('/tmp/pypath_module5')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded. Artifact dir:', ARTIFACT_DIR)

## 1. Train a Production Pipeline

Before we can save or serve a model, we need a trained object. We build a complete `Pipeline` so that preprocessing and the model are always serialised together.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = list(data.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, pipeline.predict(X_train))
test_acc  = accuracy_score(y_test,  pipeline.predict(X_test))
test_auc  = roc_auc_score(y_test,   pipeline.predict_proba(X_test)[:, 1])

print('Pipeline trained successfully.')
print(f'  Train accuracy: {train_acc:.4f}')
print(f'  Test  accuracy: {test_acc:.4f}')
print(f'  Test  AUC-ROC:  {test_auc:.4f}')

## 2. Saving and Loading Models — joblib vs pickle

| Method | Pros | Cons |
|---|---|---|
| `joblib.dump` | Faster for large numpy arrays; memory-mapped loading | joblib dependency |
| `pickle` | Standard library; no extra dependency | Slower on large arrays; less secure |

**Always save the full Pipeline**, not just the estimator. This ensures preprocessing transformers (scaler, encoder, imputer) are applied consistently at inference time.

In [ ]:
# --- joblib ---
joblib_path = ARTIFACT_DIR / 'pipeline_v1.joblib'
joblib.dump(pipeline, joblib_path)
print(f'Model saved with joblib: {joblib_path}  ({joblib_path.stat().st_size / 1024:.1f} KB)')

loaded_joblib = joblib.load(joblib_path)
joblib_acc = accuracy_score(y_test, loaded_joblib.predict(X_test))
print(f'Loaded joblib model accuracy: {joblib_acc:.4f}')

# --- pickle ---
pickle_path = ARTIFACT_DIR / 'pipeline_v1.pkl'
with open(pickle_path, 'wb') as f:
    pickle.dump(pipeline, f)
print(f'\nModel saved with pickle:  {pickle_path}  ({pickle_path.stat().st_size / 1024:.1f} KB)')

with open(pickle_path, 'rb') as f:
    loaded_pickle = pickle.load(f)
pickle_acc = accuracy_score(y_test, loaded_pickle.predict(X_test))
print(f'Loaded pickle model accuracy: {pickle_acc:.4f}')

print('\nBoth methods reproduce identical predictions:', joblib_acc == pickle_acc)

## 3. Versioning Models with MLflow

MLflow is an open-source platform for the complete ML lifecycle:
- **Tracking** — log parameters, metrics, and artifacts per run
- **Projects** — package code in a reproducible format
- **Models** — standard packaging for deployment
- **Registry** — model versioning and lifecycle management

Even without a remote server, MLflow stores everything in a local `mlruns/` directory.

In [ ]:
mlflow.set_tracking_uri(f'file://{ARTIFACT_DIR}/mlruns')
mlflow.set_experiment('breast_cancer_classification')

params = {
    'n_estimators': 100,
    'scaler':       'StandardScaler',
    'test_size':    0.2,
    'random_state': 42
}

with mlflow.start_run(run_name='rf_baseline') as run:
    # Log hyperparameters
    mlflow.log_params(params)

    # Log metrics
    mlflow.log_metric('train_accuracy', train_acc)
    mlflow.log_metric('test_accuracy',  test_acc)
    mlflow.log_metric('test_auc_roc',   test_auc)

    # Log the sklearn model
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path='model',
        input_example=X_test[:3]
    )

    # Log the serialised file as artefact
    mlflow.log_artifact(str(joblib_path))

    run_id = run.info.run_id
    print(f'MLflow run ID: {run_id}')

print('\nLogging complete. Retrieving logged run info:')
run_info = mlflow.get_run(run_id)
print('  Params: ', run_info.data.params)
print('  Metrics:', {k: round(v, 4) for k, v in run_info.data.metrics.items()})

## 4. Comparing Multiple MLflow Runs

In [ ]:
# Train and log two more variants
variants = [
    {'n_estimators': 50,  'max_depth': 5, 'run_name': 'rf_shallow'},
    {'n_estimators': 200, 'max_depth': None, 'run_name': 'rf_deep'},
]

for v in variants:
    pipe_v = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    RandomForestClassifier(
            n_estimators=v['n_estimators'],
            max_depth=v['max_depth'],
            random_state=42, n_jobs=-1
        ))
    ])
    pipe_v.fit(X_train, y_train)
    v_acc = accuracy_score(y_test, pipe_v.predict(X_test))
    v_auc = roc_auc_score(y_test, pipe_v.predict_proba(X_test)[:, 1])

    with mlflow.start_run(run_name=v['run_name']):
        mlflow.log_params({'n_estimators': v['n_estimators'], 'max_depth': str(v['max_depth'])})
        mlflow.log_metric('test_accuracy', v_acc)
        mlflow.log_metric('test_auc_roc',  v_auc)
        mlflow.sklearn.log_model(pipe_v, artifact_path='model')
    print(f"{v['run_name']:15s}  acc={v_acc:.4f}  auc={v_auc:.4f}")

# Retrieve all runs from the experiment
exp = mlflow.get_experiment_by_name('breast_cancer_classification')
runs_df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
print('\nAll MLflow runs:')
print(runs_df[['tags.mlflow.runName', 'metrics.test_accuracy', 'metrics.test_auc_roc',
               'params.n_estimators', 'params.max_depth']].to_string(index=False))

## 5. Loading a Model from MLflow

You can load any logged model by its run ID. This is the basis for model registry and promotion workflows.

In [ ]:
# Load the baseline run model by run ID
model_uri = f'runs:/{run_id}/model'
loaded_mlflow_model = mlflow.sklearn.load_model(model_uri)

mlflow_acc = accuracy_score(y_test, loaded_mlflow_model.predict(X_test))
print(f'MLflow-loaded model accuracy: {mlflow_acc:.4f}')
print('Same as original:', np.allclose(
    pipeline.predict_proba(X_test),
    loaded_mlflow_model.predict_proba(X_test)
))

## 6. Data Drift Detection

Data drift occurs when the statistical properties of the input data change between training and serving. Two common detection methods:

- **Kolmogorov-Smirnov (KS) test** — non-parametric test comparing two distributions; significant p-value (< 0.05) suggests drift
- **Population Stability Index (PSI)** — industry standard; PSI < 0.1 stable, 0.1-0.2 moderate drift, > 0.2 significant drift

In [ ]:
# Simulate reference data (training distribution)
X_reference = X_train.copy()

# Simulate drifted production data (shift mean by 2 std for features 0-4)
X_drifted = X_test.copy()
X_drifted[:, :5] += 2 * X_drifted[:, :5].std(axis=0)

# KS Test per feature
print('Kolmogorov-Smirnov Drift Detection:')
print(f'{"Feature":40s} {"KS Stat":>10s} {"p-value":>10s} {"Drift?":>8s}')
print('-' * 70)
for i, fname in enumerate(feature_names):
    ks_stat, p_val = stats.ks_2samp(X_reference[:, i], X_drifted[:, i])
    drift = 'YES' if p_val < 0.05 else 'no'
    flag  = '  <<<' if drift == 'YES' else ''
    print(f'{fname:40s} {ks_stat:>10.4f} {p_val:>10.4f} {drift:>8s}{flag}')

In [ ]:
def compute_psi(expected, actual, buckets=10):
    """Compute Population Stability Index between two arrays."""
    min_val = min(expected.min(), actual.min())
    max_val = max(expected.max(), actual.max())
    bins = np.linspace(min_val, max_val, buckets + 1)

    exp_counts = np.histogram(expected, bins=bins)[0]
    act_counts = np.histogram(actual,   bins=bins)[0]

    exp_pct = (exp_counts + 1e-6) / (len(expected) + 1e-6 * buckets)
    act_pct = (act_counts + 1e-6) / (len(actual)   + 1e-6 * buckets)

    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi

print('Population Stability Index (PSI):')
print(f'{"Feature":40s} {"PSI":>8s} {"Status":>12s}')
print('-' * 62)
for i, fname in enumerate(feature_names[:10]):
    psi = compute_psi(X_reference[:, i], X_drifted[:, i])
    status = 'DRIFT' if psi > 0.2 else ('Moderate' if psi > 0.1 else 'Stable')
    print(f'{fname:40s} {psi:>8.4f} {status:>12s}')

## 7. Drift Report with Evidently

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently import ColumnMapping

# Build DataFrames
ref_df  = pd.DataFrame(X_reference, columns=feature_names)
curr_df = pd.DataFrame(X_drifted,   columns=feature_names)

# Create and run the drift report
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(reference_data=ref_df, current_data=curr_df)

# Save HTML report
report_path = ARTIFACT_DIR / 'drift_report.html'
drift_report.save_html(str(report_path))
print(f'Drift report saved: {report_path}')

# Extract and display summary
drift_result = drift_report.as_dict()
try:
    drift_summary = drift_result['metrics'][0]['result']
    print(f'\nDataset drift detected: {drift_summary.get("dataset_drift", "N/A")}')
    print(f'Drifted columns:        {drift_summary.get("number_of_drifted_columns", "N/A")}')
except Exception:
    print('\nDrift report generated successfully (view HTML for details).')

## 8. FastAPI Model Serving

The most common pattern for serving an ML model is to wrap it in a REST API. FastAPI is a modern, high-performance Python web framework ideal for this purpose.

The `%%writefile` magic writes the following code to `app.py`, which can then be launched with `uvicorn`.

In [ ]:
%%writefile /tmp/pypath_module5/app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import joblib
import numpy as np
from pathlib import Path

# ---- App initialisation ----
app = FastAPI(
    title='PyPath ML API',
    description='Breast Cancer Classification Model',
    version='1.0.0'
)

# ---- Load model at startup ----
MODEL_PATH = Path('/tmp/pypath_module5/pipeline_v1.joblib')

@app.on_event('startup')
def load_model():
    global model
    model = joblib.load(MODEL_PATH)
    print(f'Model loaded from {MODEL_PATH}')

# ---- Request / Response schemas ----
class PredictRequest(BaseModel):
    features: List[float]  # 30 feature values for one sample

class PredictResponse(BaseModel):
    prediction: int           # 0 = malignant, 1 = benign
    probability_benign: float
    probability_malignant: float

# ---- Endpoints ----
@app.get('/')
def root():
    return {'status': 'ok', 'model': 'RandomForest Breast Cancer Classifier v1.0'}

@app.get('/health')
def health():
    return {'status': 'healthy'}

@app.post('/predict', response_model=PredictResponse)
def predict(request: PredictRequest):
    if len(request.features) != 30:
        raise HTTPException(status_code=422,
                            detail=f'Expected 30 features, got {len(request.features)}')
    X = np.array(request.features).reshape(1, -1)
    prediction  = int(model.predict(X)[0])
    proba       = model.predict_proba(X)[0]
    return PredictResponse(
        prediction=prediction,
        probability_benign=float(proba[1]),
        probability_malignant=float(proba[0])
    )

@app.post('/predict_batch')
def predict_batch(data: List[PredictRequest]):
    X = np.array([r.features for r in data])
    predictions = model.predict(X).tolist()
    probabilities = model.predict_proba(X)[:, 1].tolist()
    return {'predictions': predictions, 'probabilities_benign': probabilities}

In [ ]:
# Verify the file was written
with open('/tmp/pypath_module5/app.py') as f:
    lines = f.readlines()
print(f'app.py written: {len(lines)} lines')

# Show how to run it
print('\nTo run locally:')
print('  pip install fastapi uvicorn')
print('  uvicorn app:app --host 0.0.0.0 --port 8000 --reload')
print()
print('To test:')
print('  curl -X POST http://localhost:8000/predict \\')
print('    -H "Content-Type: application/json" \\')
print('    -d \'{"features": [17.99, 10.38, 122.8, 1001.0, ...]}\'  # 30 values')

## 9. Dockerfile Overview

To containerise the serving API, create a `Dockerfile` in the same directory as `app.py`:

```dockerfile
# Base image — slim Python for smaller container size
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code and model
COPY app.py .
COPY pipeline_v1.joblib .

# Expose the API port
EXPOSE 8000

# Health check — Docker will restart container if this fails
HEALTHCHECK --interval=30s --timeout=10s CMD curl -f http://localhost:8000/health || exit 1

# Start the server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

**requirements.txt:**
```
fastapi==0.111.0
uvicorn[standard]==0.29.0
scikit-learn==1.4.2
numpy==1.26.4
joblib==1.4.0
pydantic==2.7.1
```

**Build and run:**
```bash
docker build -t pypath-ml-api:v1 .
docker run -p 8000:8000 pypath-ml-api:v1
```

In [ ]:
%%writefile /tmp/pypath_module5/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY pipeline_v1.joblib .

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile /tmp/pypath_module5/requirements.txt
fastapi==0.111.0
uvicorn[standard]==0.29.0
scikit-learn==1.4.2
numpy==1.26.4
joblib==1.4.0
pydantic==2.7.1

In [ ]:
# Summary of all artefacts produced
print('=== Production Artefacts Created ===')
for path in sorted(ARTIFACT_DIR.rglob('*')):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f'  {path.relative_to(ARTIFACT_DIR)}  ({size_kb:.1f} KB)')

print('\n=== Module 5 Complete ===')
print('You have learned:')
print('  1. ML Concepts — bias-variance, learning curves')
print('  2. Sklearn Basics — Pipeline, ColumnTransformer, GridSearch')
print('  3. Regression — Linear, Ridge, Lasso, ElasticNet')
print('  4. Classification — LR, Trees, RF, SVM, GradBoost')
print('  5. Clustering — KMeans, DBSCAN, Hierarchical')
print('  6. Model Evaluation — ROC, PR, CV, imbalanced data')
print('  7. Feature Engineering — encoders, scaling, PCA, selection')
print('  8. Ensemble Methods — XGBoost, LGBM, Stacking, Blending')
print('  9. ML Production — joblib, MLflow, drift detection, FastAPI')

## Practice Exercises

**Exercise 1 — MLflow Experiment Comparison**
Train five different classifiers on `load_breast_cancer()` (LogisticRegression, RandomForest, GradientBoosting, XGBoost, LightGBM). Log each as a separate MLflow run with parameters and metrics (accuracy, AUC-ROC). Use `mlflow.search_runs()` to retrieve all results into a DataFrame and create a bar chart comparing test AUC-ROC across runs.

**Exercise 2 — Drift Detection on Feature Subsets**
Using the drift detection code from section 6, create a function `detect_drift(X_ref, X_curr, feature_names, alpha=0.05)` that returns a DataFrame of drifted features only. Then simulate three scenarios: (a) no drift, (b) drift in numeric features only, (c) drift in all features. Verify your function correctly identifies each scenario.

**Exercise 3 — API Integration Test**
Extend `app.py` to add a `/metrics` endpoint that returns the model's training metadata (accuracy, AUC-ROC, number of features, feature names). Add a `/feature_names` endpoint that returns the expected input feature order. Write a Python script that calls these two endpoints using the `requests` library and pretty-prints the results. (Note: you will need to start the server in a separate terminal to test live.)